## 1. Загрузка данных

In [ ]:
import sys
sys.path.append("..")

import torch
from torch.utils.data import DataLoader

from src.data_loader import load_pkl, load_or_build_cache
from src.features import extract_sequence_features
from src.advanced_models import BiLSTMModel, CNN1DModel, TransformerModel, count_parameters
from src.train import (MOSEISequenceDataset, fit_sequence, evaluate_sequence,
                        compute_sentiment_class_weights, compute_emotion_pos_weights)

In [ ]:
pkl_data = load_pkl()
label_cache = load_or_build_cache()
seq_features = extract_sequence_features(pkl_data)

for modality in ["text", "audio"]:
    X_shape = seq_features[modality]["train"]["X"].shape
    mask_shape = seq_features[modality]["train"]["mask"].shape
    print(f"{modality}: X = {X_shape}, mask = {mask_shape}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Устройство:", device)

## 2. Проверочный прогон

In [ ]:
for modality, dim in [("text", 300), ("audio", 74)]:
    print(f"\n--- {modality} (input_dim={dim}) ---")
    dummy_x = torch.randn(4, 50, dim).to(device)
    dummy_mask = torch.ones(4, 50, dtype=torch.bool).to(device)
    dummy_mask[2, 30:] = False

    for ModelCls, name in [(BiLSTMModel, "BiLSTM"), (CNN1DModel, "CNN1D"), (TransformerModel, "Transformer")]:
        model = ModelCls(input_dim=dim).to(device)
        sent_out, emo_out = model(dummy_x, dummy_mask)
        print(f"  {name:12s} | sent={tuple(sent_out.shape)} emo={tuple(emo_out.shape)} | params={count_parameters(model):,}")

In [ ]:
X_train = seq_features["text"]["train"]["X"]
mask_train = seq_features["text"]["train"]["mask"]
X_valid = seq_features["text"]["valid"]["X"]
mask_valid = seq_features["text"]["valid"]["mask"]

train_ds = MOSEISequenceDataset(X_train, mask_train,
                                 label_cache["train"]["sentiment_class"],
                                 label_cache["train"]["emotion_binary"],
                                 label_cache["train"]["mask"])
valid_ds = MOSEISequenceDataset(X_valid, mask_valid,
                                 label_cache["valid"]["sentiment_class"],
                                 label_cache["valid"]["emotion_binary"],
                                 label_cache["valid"]["mask"])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False)

sent_weights = compute_sentiment_class_weights(label_cache["train"]["sentiment_class"])
emo_weights = compute_emotion_pos_weights(label_cache["train"]["emotion_binary"], label_cache["train"]["mask"])

model = TransformerModel(input_dim=300)
print("Модель:", type(model).__name__, "| параметров:", sum(p.numel() for p in model.parameters()))

model, history = fit_sequence(model, train_loader, valid_loader, device,
                               sent_weights, emo_weights, epochs=5, patience=3)

## 3. Обучение моделей

In [ ]:
import os
import pandas as pd

os.makedirs("../experiments", exist_ok=True)

ARCH_CONFIGS = [
    ("text", "BiLSTM", BiLSTMModel, 300),
    ("text", "CNN1D", CNN1DModel, 300),
    ("text", "Transformer", TransformerModel, 300),
    ("audio", "BiLSTM", BiLSTMModel, 74),
    ("audio", "CNN1D", CNN1DModel, 74),
    ("audio", "Transformer", TransformerModel, 74),
]

results_advanced = []
histories_advanced = {}
trained_models_advanced = {}

for modality, arch_name, ModelCls, input_dim in ARCH_CONFIGS:
    print(f"\n{'='*60}\n{modality} / {arch_name}\n{'='*60}")

    X_train, mask_train = seq_features[modality]["train"]["X"], seq_features[modality]["train"]["mask"]
    X_valid, mask_valid = seq_features[modality]["valid"]["X"], seq_features[modality]["valid"]["mask"]
    X_test, mask_test = seq_features[modality]["test"]["X"], seq_features[modality]["test"]["mask"]

    train_ds = MOSEISequenceDataset(X_train, mask_train,
                                     label_cache["train"]["sentiment_class"],
                                     label_cache["train"]["emotion_binary"],
                                     label_cache["train"]["mask"])
    valid_ds = MOSEISequenceDataset(X_valid, mask_valid,
                                     label_cache["valid"]["sentiment_class"],
                                     label_cache["valid"]["emotion_binary"],
                                     label_cache["valid"]["mask"])
    test_ds = MOSEISequenceDataset(X_test, mask_test,
                                    label_cache["test"]["sentiment_class"],
                                    label_cache["test"]["emotion_binary"],
                                    label_cache["test"]["mask"])

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

    sent_weights = compute_sentiment_class_weights(label_cache["train"]["sentiment_class"])
    emo_weights = compute_emotion_pos_weights(label_cache["train"]["emotion_binary"], label_cache["train"]["mask"])

    model = ModelCls(input_dim=input_dim)
    n_params = count_parameters(model)

    model, history = fit_sequence(model, train_loader, valid_loader, device,
                                   sent_weights, emo_weights, epochs=30, patience=5, verbose=False)

    sm_test, em_test = evaluate_sequence(model, test_loader, device)

    results_advanced.append({
        "modality": modality,
        "architecture": arch_name,
        "input_dim": input_dim,
        "n_params": n_params,
        "epochs_trained": len(history),
        "sentiment_macro_f1": round(sm_test["macro_f1_3class"], 3),
        "sentiment_acc2": round(sm_test["acc2"], 3),
        "emotion_macro_f1": round(em_test["macro_f1"], 3),
        "emotion_mean_auc": round(em_test["mean_auc"], 3),
    })
    histories_advanced[f"{modality}_{arch_name}"] = history
    trained_models_advanced[f"{modality}_{arch_name}"] = model

    print(f"TEST: sentiment macro-F1={sm_test['macro_f1_3class']:.3f} | "
          f"emotion macro-F1={em_test['macro_f1']:.3f} | epochs={len(history)}")

results_advanced_df = pd.DataFrame(results_advanced)
results_advanced_df.to_csv("../experiments/results_advanced.csv", index=False)
results_advanced_df

## 4. Подбор гиперпараметров

In [ ]:
import os
import itertools
import pandas as pd

os.makedirs("../experiments", exist_ok=True)

GRID = {
    "d_model": [64, 128],
    "num_layers": [1, 2],
    "dropout": [0.1, 0.3],
    "lr": [1e-3, 5e-4],
}

X_train, mask_train = seq_features["text"]["train"]["X"], seq_features["text"]["train"]["mask"]
X_valid, mask_valid = seq_features["text"]["valid"]["X"], seq_features["text"]["valid"]["mask"]

train_ds = MOSEISequenceDataset(X_train, mask_train,
                                 label_cache["train"]["sentiment_class"],
                                 label_cache["train"]["emotion_binary"],
                                 label_cache["train"]["mask"])
valid_ds = MOSEISequenceDataset(X_valid, mask_valid,
                                 label_cache["valid"]["sentiment_class"],
                                 label_cache["valid"]["emotion_binary"],
                                 label_cache["valid"]["mask"])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False)

sent_weights = compute_sentiment_class_weights(label_cache["train"]["sentiment_class"])
emo_weights = compute_emotion_pos_weights(label_cache["train"]["emotion_binary"], label_cache["train"]["mask"])

search_results = []
search_models = {}

combos = list(itertools.product(*GRID.values()))
for i, (d_model, num_layers, dropout, lr) in enumerate(combos, 1):
    print(f"[{i}/{len(combos)}] d_model={d_model} num_layers={num_layers} dropout={dropout} lr={lr}")

    model = TransformerModel(input_dim=300, d_model=d_model, num_layers=num_layers, dropout=dropout)
    model, history = fit_sequence(model, train_loader, valid_loader, device,
                                   sent_weights, emo_weights, epochs=30, patience=5,
                                   lr=lr, verbose=False)

    best_val_f1 = max(h["val_sentiment_macro_f1"] for h in history)
    config_key = f"d{d_model}_l{num_layers}_dr{dropout}_lr{lr}"

    search_results.append({
        "d_model": d_model, "num_layers": num_layers, "dropout": dropout, "lr": lr,
        "epochs_trained": len(history), "best_val_sentiment_macro_f1": round(best_val_f1, 4),
    })
    search_models[config_key] = model
    print(f"    best val macro-F1 = {best_val_f1:.4f} (epochs={len(history)})")

search_df = pd.DataFrame(search_results).sort_values("best_val_sentiment_macro_f1", ascending=False)
search_df.to_csv("../experiments/hyperparam_search.csv", index=False)
search_df

In [ ]:
best_row = search_df.iloc[0]
best_key = f"d{int(best_row.d_model)}_l{int(best_row.num_layers)}_dr{best_row.dropout}_lr{best_row.lr}"
best_model = search_models[best_key]

X_test, mask_test = seq_features["text"]["test"]["X"], seq_features["text"]["test"]["mask"]
test_ds = MOSEISequenceDataset(X_test, mask_test,
                                label_cache["test"]["sentiment_class"],
                                label_cache["test"]["emotion_binary"],
                                label_cache["test"]["mask"])
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

sm_test, em_test = evaluate_sequence(best_model, test_loader, device)
print("Лучшая конфигурация:", best_row.to_dict())
print(f"TEST: sentiment macro-F1={sm_test['macro_f1_3class']:.3f} | "
      f"emotion macro-F1={em_test['macro_f1']:.3f} | mean-AUC={em_test['mean_auc']:.3f}")

In [ ]:
for param in ["d_model", "num_layers", "dropout", "lr"]:
    means = search_df.groupby(param)["best_val_sentiment_macro_f1"].mean()
    spread = means.max() - means.min()
    print(f"\n{param} (разброс влияния: {spread:.4f}):")
    print(means.round(4))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
params = ["d_model", "num_layers", "dropout", "lr"]

for ax, param in zip(axes, params):
    means = search_df.groupby(param)["best_val_sentiment_macro_f1"].mean()
    ax.bar(means.index.astype(str), means.values, color="#534AB7")
    ax.set_title(param)
    ax.set_ylim(0.55, 0.59)
    ax.set_ylabel("val macro-F1" if param == "d_model" else "")

plt.suptitle("Влияние гиперпараметров на качество Transformer (текст)")
plt.tight_layout()
plt.savefig("../figures/architectures/hyperparam_impact.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import os

os.makedirs("../figures/architectures", exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 10))
ax.set_xlim(0, 10)
ax.set_ylim(0, 14.5)
ax.axis("off")

def box(x, y, w, h, title, subtitle, color, edge):
    rect = mpatches.FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                     linewidth=1.3, edgecolor=edge, facecolor=color)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h*0.64, title, ha="center", va="center", fontsize=11, fontweight="bold")
    ax.text(x + w/2, y + h*0.28, subtitle, ha="center", va="center", fontsize=9, color="#333")

def arrow(x1, y1, x2, y2, color="#5F5E5A"):
    a = FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=14,
                          linewidth=1.2, color=color)
    ax.add_patch(a)

GRAY_FILL, GRAY_EDGE = "#F1EFE8", "#5F5E5A"
PURPLE_FILL, PURPLE_EDGE = "#EEEDFE", "#534AB7"
TEAL_FILL, TEAL_EDGE = "#E1F5EE", "#0F6E56"

box(2, 12.6, 6, 1.3, "Входная последовательность", "(B, T=50, D), D = 300 или 74", GRAY_FILL, GRAY_EDGE)
arrow(5, 12.6, 5, 12.1)
box(2, 10.8, 6, 1.3, "Линейная проекция", "D → d_model = 128", GRAY_FILL, GRAY_EDGE)
arrow(5, 10.8, 5, 10.3)
box(1.7, 9.0, 6.6, 1.3, "Transformer encoder layer 1", "self-attention (4 головы) + FFN, dropout 0.1", PURPLE_FILL, PURPLE_EDGE)
arrow(5, 9.0, 5, 8.5)
box(1.7, 7.2, 6.6, 1.3, "Transformer encoder layer 2", "self-attention (4 головы) + FFN, dropout 0.1", PURPLE_FILL, PURPLE_EDGE)
arrow(5, 7.2, 5, 6.7)
ax.text(7.0, 6.95, "маска паддинга", fontsize=9, color="#5F5E5A")
box(2, 5.4, 6, 1.3, "Masked mean pooling", "по реальным шагам → (B, 128)", GRAY_FILL, GRAY_EDGE)

arrow(5, 5.4, 2.8, 4.1, color=PURPLE_EDGE)
arrow(5, 5.4, 7.2, 4.1, color=TEAL_EDGE)

box(1, 2.8, 3.6, 1.3, "Sentiment head", "128 → 128 → 3 класса", PURPLE_FILL, PURPLE_EDGE)
box(5.4, 2.8, 3.6, 1.3, "Emotion head", "128 → 128 → 6 меток", TEAL_FILL, TEAL_EDGE)

plt.tight_layout()
plt.savefig("../figures/architectures/transformer_architecture.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd

final_text_transformer = pd.DataFrame([{
    "modality": "text", "architecture": "Transformer (default)",
    "d_model": 128, "num_layers": 2, "dropout": 0.3, "lr": 1e-3,
    "sentiment_macro_f1": 0.579, "emotion_macro_f1": 0.402,
}, {
    "modality": "text", "architecture": "Transformer (tuned)",
    "d_model": 128, "num_layers": 2, "dropout": 0.1, "lr": 5e-4,
    "sentiment_macro_f1": 0.602, "emotion_macro_f1": 0.413,
}])
final_text_transformer.to_csv("../experiments/transformer_before_after_tuning.csv", index=False)
final_text_transformer

## Выводы

Для каждой из двух модальностей (текст, аудио) построены три архитектуры энкодера последовательностей: двунаправленный LSTM, одномерная свёртка с тремя параллельными ветвями (размеры ядра 3/5/7), и Transformer-энкодер с self-attention. Все шесть моделей используют единый multitask-интерфейс с двумя задачными головами, идентичный baseline.
С дефолтными гиперпараметрами усложнение архитектуры не дало однозначного превосходства над baseline: лучший результат на тексте (CNN1D, sentiment macro-F1 = 0,583; Transformer = 0,579) оказался немного ниже лучшего baseline (GloVe FFN, 0,589). BiLSTM продемонстрировал нестабильность обучения на обеих модальностях (early stopping срабатывал на 1–2-й эпохе из-за отсутствия улучшения), которая частично, но не полностью устранялась применением gradient clipping.
Подбор гиперпараметров для конфигурации Transformer на текстовой модальности (сетка из 16 комбинаций: d_model ∈ {64, 128}, num_layers ∈ {1, 2}, dropout ∈ {0.1, 0.3}, lr ∈ {1·10⁻³, 5·10⁻⁴}) выявил, что наибольшее влияние на качество оказывают скорость обучения и глубина энкодера (разброс среднего val-F1 по уровням — 0,0029 и 0,0027 соответственно), тогда как ширина модели (d_model) и dropout в исследованном диапазоне практически не влияли на результат (разброс 0,0005 и 0,0008). Лучшая конфигурация (d_model=128, num_layers=2, dropout=0.1, lr=5·10⁻⁴) на тестовой выборке показала sentiment macro-F1 = 0,602 и emotion macro-F1 = 0,413, превзойдя как дефолтную конфигурацию Transformer (0,579 / 0,402), так и лучший результат baseline (0,589 / 0,409).
Полученный результат подтверждает, что усложнение архитектуры даёт выигрыш только при сопутствующем подборе гиперпараметров — без него Transformer уступал простому усреднению признаков на feed-forward сети. Это определяет практическую рекомендацию для раздела 6: объединение модальностей целесообразно строить на базе настроенного Transformer-энкодера текста, а не его дефолтной конфигурации.